In [6]:
!pip3 install  -q datasets seqeval


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip3 install --upgrade pip


In [62]:
import numpy as np
import pandas as pd
import random
import math
import pickle
from collections import Counter, defaultdict

from datasets import load_dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

random.seed(42)
np.random.seed(42)

In [32]:
#Loading the dataset
raw =load_dataset("eriktks/conll2003", revision="refs/convert/parquet")
print(raw)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [33]:
# Getting the ClassLabel from ner_tags
Labels = raw["train"].features["ner_tags"].feature.names
print("Total No of tags: ",len(Labels))
print(Labels)


Total No of tags:  9
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


Dropping the pos_tags and chunk_tags as they are not need For NER Tagging

In [38]:
# Loading the ClassLabel
NER_Feature = raw["train"].features["ner_tags"].feature

# Every sentence in data is converted to token and tag pairs
def pairing(split):
    sentences = []
    for sen in raw[split]:
        tokens = list(sen["tokens"])
        tags = []
        for tag in sen["ner_tags"]:
            tags.append(NER_Feature.int2str(tag))
        sentences.append((tokens,tags))
    
    return sentences
        

sets = ["train","validation","test"]
data = {split:pairing(split) for split in sets}

In [42]:
# examing one sentence in train set
tokens,tags = data["train"][3]
for w,t in zip(tokens,tags):
    print(f"{w} {t}")

The O
European B-ORG
Commission I-ORG
said O
on O
Thursday O
it O
disagreed O
with O
German B-MISC
advice O
to O
consumers O
to O
shun O
British B-MISC
lamb O
until O
scientists O
determine O
whether O
mad O
cow O
disease O
can O
be O
transmitted O
to O
sheep O
. O


In [ ]:
# if token length is zero or sentence starting with -DOCSTART- those token,tag pairs are dropped from data
def cleaning_data(sentences):
    drop = 0
    for tokens,tags in sentences:
        if len(tokens)==0 or tokens[0] == "-DOCSTART-":
            drop += 1
        
    return drop

for split in data:
    dropped = cleaning_data(data[split])
    print(f"{split}: {dropped} dropped")

train: 0 dropped
validation: 0 dropped
test: 0 dropped


In [49]:
train_vocabulary = Counter([w for tok,_ in data["train"] for w in tok])
test_vocabulary = [w for tok,_ in data["test"] for w in tok]
print(f"Training vocabulary size: {len(train_vocabulary)}")

out_of_vocab = sum([w not in train_vocabulary for w in test_vocabulary])
print(f"Total no of out of vocabulary words is:{out_of_vocab}")


Training vocabulary size: 23623
Total no of out of vocabulary words is:5656


In [55]:
# Deciding some logical divisions if the word is out of vocabulary word
def word_shape(word):
    if any(ch.isdigit() for ch in word):
        return "DIGIT"
    if word.isupper() and any(ch.isalpha() for ch in word):
        return "ALLCAPS"
    if word[:1].isupper():
        return "INITCAP"
    if word.islower():
        return "LOWER"
    return "OTHER"

In [81]:
class HMM_Tagger:
    START, STOP = "<START","STOP"

    def __init__(self,tags,lamb=0.1,rare_threshold=5, max_suffix=3,open_class_tags=None):
        self.tags = list(tags)
        self.tagToId = {t: i for i,t in enumerate(self.tags)}
        self.lamb = lamb
        self.rare_threshold = rare_threshold
        self.max_suffix = max_suffix
        self.open_class_tags = list(open_class_tags)
        # what is the unknown word fallback is used
        self.unk_stats = Counter()

    def fit(self,sentences):
        no_of_tags = len(self.tags)
        lamb = self.lamb
        
        transitions = np.zeros((no_of_tags+1,no_of_tags+1))
        start_row, stop_col = no_of_tags,no_of_tags
        emit_counts = defaultdict(Counter)
        word_frequency = Counter()

        for tokens, tags in sentences:
            prev = start_row
            for w,t in zip(tokens,tags):
                j = self.tagToId[t]
                transitions[prev][j] += 1
                emit_counts[t][w] += 1
                word_frequency[w] += 1
                prev = j
            transitions[prev][stop_col] += 1

        #adding lambda smoothing and then log

        transition_prob = (transitions + lamb)/(transitions.sum(axis=1,keepdims=True)+ (no_of_tags+1)*lamb)
        # tag to tag
        self.log_transitions = np.log(transition_prob[:no_of_tags,:no_of_tags])
        # start to tag
        self.log_start = np.log(transition_prob[start_row,:no_of_tags])
        # tag to stop
        self.log_stop = np.log(transition_prob[:no_of_tags,stop_col])

        # emissions for already known words
        self.vocab = set(word_frequency)
        vocab_length = len(self.vocab)
        self.tag_totals = np.array([sum(emit_counts[t].values()) for t in self.tags], dtype=float)
        self.emit_denominator = self.tag_totals + lamb * (vocab_length + 1)
        self.emit_counts = emit_counts

        # tag prior needed for Bayes inversion in suffix model
        self.log_tag_prior = np.log(self.tag_totals / self.tag_totals.sum())

        # suffix model that is built only from rare words
        self.suffix_counts = defaultdict(lambda: np.zeros(no_of_tags))
        for t in self.tags:
            j = self.tagToId[t]
            for w, c in emit_counts[t].items():
                if word_frequency[w] > self.rare_threshold:
                    continue
                shape = word_shape(w)
                for k in range(1, self.max_suffix + 1):
                    if len(w) >= k:
                        self.suffix_counts[(w[-k:].lower(), shape)][j] += c
        return self

    # emissions

    def emission_known(self,word):
        counts = np.array([self.emit_counts[t][word] for t in self.tags],dtype=float)
        return np.log((counts+self.lamb)/self.emit_denominator)
        
    def emission_unknown(self, word):
            T = len(self.tags)
            shape = word_shape(word)
            for k in range(min(self.max_suffix, len(word)), 0, -1):
                key = (word[-k:].lower(), shape)
                if key in self.suffix_counts:
                    counts = self.suffix_counts[key]
                    p_tag_given_suffix = (counts + self.lamb) / (counts.sum() + self.lamb * T)
                    self.unk_stats[f"suffix-{k}"] += 1
                    return np.log(p_tag_given_suffix) - self.log_tag_prior

            # nothing matched then uniform over open-class tags (practically) zero for the rest
            self.unk_stats["uniform"] += 1
            scores = np.full(T, -1e9)
            for t in self.open_class_tags:
                scores[self.tagToId[t]] = -math.log(len(self.open_class_tags))
            return scores

    def emission_matrix(self, tokens):
        return np.stack([
            self.emission_known(w) if w in self.vocab else self.emission_unknown(w)
            for w in tokens
        ])
    
    def viterbi(self, tokens):
        n, T = len(tokens), len(self.tags)
        emit = self.emission_matrix(tokens)        
        score = np.empty((n, T))
        back = np.zeros((n, T), dtype=int)

        score[0] = self.log_start + emit[0]
        for i in range(1, n):
            # cand[prev, cur] = best score ending in prev + log P(cur | prev)
            cand = score[i - 1][:, None] + self.log_transitions
            back[i] = cand.argmax(axis=0)
            score[i] = cand.max(axis=0) + emit[i]

        final = score[-1] + self.log_stop
        best = [int(final.argmax())]
        for i in range(n - 1, 0, -1):
            best.append(int(back[i, best[-1]]))
        best.reverse()
        return [self.tags[j] for j in best]

    def predict(self, sentences):
        return [self.viterbi(tokens) for tokens, _ in sentences]

In [82]:
OPEN_CLASS_TAGS = Labels    
hidden_markov_model = HMM_Tagger(Labels, lamb=0.1, max_suffix=3, rare_threshold=5,
                open_class_tags=OPEN_CLASS_TAGS).fit(data["train"])

print("Size of the vocabulary:", len(hidden_markov_model.vocab))
print("length of the suffix entries  :", len(hidden_markov_model.suffix_counts))

Size of the vocabulary: 23623
length of the suffix entries  : 6682


In [73]:
transitions_df = pd.DataFrame(np.exp(hidden_markov_model.log_transitions),index=Labels,columns=Labels)
print("P(next tag/previous tag), rows are previous tag")
print(transitions_df.round(3))
print()
start_df = pd.Series(np.exp(hidden_markov_model.log_start),index=Labels,name='P(tag/start)')
print(start_df.round(3).to_frame())

ner_tags = ["B-PER", "B-ORG", "B-LOC", "B-MISC"]
print()
for tag in ner_tags:
    print(tag,[w for w,_ in hidden_markov_model.emit_counts[tag].most_common(10)])

P(next tag/previous tag), rows are previous tag
            O  B-PER  I-PER  B-ORG  I-ORG  B-LOC  I-LOC  B-MISC  I-MISC
O       0.819  0.031  0.000  0.022  0.000  0.033  0.000   0.017   0.000
B-PER   0.340  0.000  0.649  0.000  0.000  0.000  0.000   0.000   0.000
I-PER   0.907  0.000  0.054  0.000  0.000  0.000  0.000   0.000   0.000
B-ORG   0.586  0.000  0.000  0.003  0.393  0.000  0.000   0.001   0.000
I-ORG   0.648  0.001  0.000  0.002  0.329  0.000  0.000   0.001   0.000
B-LOC   0.832  0.000  0.000  0.001  0.000  0.001  0.146   0.004   0.000
I-LOC   0.850  0.000  0.000  0.001  0.000  0.003  0.100   0.003   0.000
B-MISC  0.706  0.017  0.000  0.009  0.000  0.002  0.000   0.009   0.250
I-MISC  0.627  0.003  0.000  0.006  0.000  0.002  0.000   0.004   0.257

        P(tag/start)
O              0.581
B-PER          0.096
I-PER          0.000
B-ORG          0.175
I-ORG          0.000
B-LOC          0.113
I-LOC          0.000
B-MISC         0.036
I-MISC         0.000

B-PER ['Clinton', 'M

In [85]:
def evaluation(model, sentences, name):
    model.unk_stats.clear()
    truth = [tags for _, tags in sentences]
    predictions = model.predict(sentences)

    p = precision_score(truth, predictions)
    r = recall_score(truth, predictions)
    f = f1_score(truth, predictions)

    flat_truth = [t for s in truth for t in s]
    flat_predictions = [t for s in predictions for t in s]
    accuracy = np.mean([g == q for g, q in zip(flat_truth, flat_predictions)])

    print("="*60)
    print(name)
    print("="*60)

    print(f"Entity-level  Precision = {p:.4f}  Recall = {r:.4f}  F1_score = {f:.4f}")
    print(f"Token accuracy  is {accuracy:.4f}")
    print("Unknown-word fallbacks used:", dict(model.unk_stats))
    print("="*60)
    print(classification_report(truth, predictions, digits=4))
    return {"precision": p, "recall": r, "f1": f, "token_acc": accuracy, "pred": predictions}

validation_res = evaluation(hidden_markov_model, data["validation"], "Validation")
test_result = evaluation(hidden_markov_model, data["test"], "Test")

Validation
Entity-level  Precision = 0.7967  Recall = 0.7876  F1_score = 0.7921
Token accuracy  is 0.9610
Unknown-word fallbacks used: {'suffix-3': 3644, 'suffix-2': 568, 'suffix-1': 73, 'uniform': 8}
              precision    recall  f1-score   support

         LOC     0.8202    0.7948    0.8073      1837
        MISC     0.8111    0.7777    0.7940       922
         ORG     0.6849    0.7472    0.7147      1341
         PER     0.8592    0.8149    0.8364      1842

   micro avg     0.7967    0.7876    0.7921      5942
   macro avg     0.7938    0.7836    0.7881      5942
weighted avg     0.8003    0.7876    0.7934      5942

Test
Entity-level  Precision = 0.6885  Recall = 0.6843  F1_score = 0.6864
Token accuracy  is 0.9370
Unknown-word fallbacks used: {'suffix-2': 901, 'suffix-3': 4595, 'suffix-1': 134, 'uniform': 26}
              precision    recall  f1-score   support

         LOC     0.6849    0.7374    0.7102      1668
        MISC     0.6218    0.6652    0.6428       702
    

In [87]:
for lam in [0.01, 0.1, 0.5, 1.0]:
    m = HMM_Tagger(Labels, lamb=lam, open_class_tags=OPEN_CLASS_TAGS).fit(data["train"])
    truth = [t for _, t in data["validation"]]
    print(f"lambda = {lam:<5} validation  F1 = {f1_score(truth, m.predict(data['validation'])):.4f}")

lambda = 0.01  validation  F1 = 0.8168
lambda = 0.1   validation  F1 = 0.7921
lambda = 0.5   validation  F1 = 0.7544
lambda = 1.0   validation  F1 = 0.7136


In [92]:
artifacts = {
    "labels": Labels,
    "log_trans": hidden_markov_model.log_transitions,
    "log_start": hidden_markov_model.log_start,
    "log_stop": hidden_markov_model.log_stop,
    "data": data,
    "task1_results": {k: v for k, v in test_result.items() if k != "pred"},
    "task1_test_pred": test_result["pred"],
}
with open("/Users/sunny/Downloads/task1_hmm.pkl", "wb") as f:
    pickle.dump(artifacts, f)
